In [4]:
import numpy as np 
import pandas as pd 


In [5]:
df = pd.read_csv('../2TextPreprocessing/IMDBDataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Text preprocessing on the review

## LowerCasing

In [6]:
df['review'][0].lower()

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show is due to the fa

In [7]:
df['review'] = df['review'].str.lower()


In [8]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [9]:
df['review'][3]

"basically there's a family where a little boy (jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />this movie is slower than a soap opera... and suddenly, jake decides to become rambo and kill the zombie.<br /><br />ok, first of all when you're going to make a film you must decide if its a thriller or a drama! as a drama the movie is watchable. parents are divorcing & arguing like in real life. and then we have jake with his closet which totally ruins all the film! i expected to see a boogeyman similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. as for the shots with jake: just ignore them."

## Removing html if present

In [10]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'',str(text))

df['review'] = df['review'].apply(remove_html_tags)

Removing url if present

In [11]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', str(text))

df['review'] = df['review'].apply(remove_url)


Removing punctuation

In [12]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [13]:
exclude = string.punctuation

In [14]:
df['review'][1]


'a wonderful little production. the filming technique is very unassuming- very old-time-bbc fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. the actors are extremely well chosen- michael sheen not only "has got all the polari" but he has all the voices down pat too! you can truly see the seamless editing guided by the references to williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. a masterful production about one of the great master\'s of comedy and his life. the realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. it plays on our knowledge and our senses, particularly with the scenes concerning orton and halliwell and the sets (particularly of their flat with halliwell\'s murals decorating every surface) are terribly well done.'

In [15]:
def remove_punc_fast(text):
    return text.translate(str.maketrans('', '', exclude))

df['review'] = df['review'].apply(remove_punc_fast)


In [16]:
df['review'][1]

'a wonderful little production the filming technique is very unassuming very oldtimebbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices down pat too you can truly see the seamless editing guided by the references to williams diary entries not only is it well worth the watching but it is a terrificly written and performed piece a masterful production about one of the great masters of comedy and his life the realism really comes home with the little things the fantasy of the guard which rather than use the traditional dream techniques remains solid then disappears it plays on our knowledge and our senses particularly with the scenes concerning orton and halliwell and the sets particularly of their flat with halliwells murals decorating every surface are terribly well done'

## ChatWord Conversion 

Not needed here but might be needed mainly on the messages kind of data

## MisSpelled Treatment

In [17]:
from textblob import TextBlob

def fix_spelling(text):
    textBlb = TextBlob(text)

    return textBlb.correct().string

fix_spelling('ceertain')


'certain'

In [ ]:
df['review'] = df['review'].apply(fix_spelling)


## Removing Stop words 

In [20]:
from nltk.corpus import stopwords

In [21]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [ ]:
df['review'] = df['review'].apply(remove_stopwords)


In [ ]:
df.sample()

,id,title,overview,genre
3281,547016,The Old Guard,four undying warriors whove secretly protected...,"['Action', 'Fantasy']"


## Tokenizations 

using spacy for tokenization

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
def tokenize(text):
    docs = nlp(text)
    tokens =[]
    for token in docs:
        tokens.append(token)
    return tokens

# print(type(tokenize("Hello what's my name ?")))
print(tokenize("Hello what's my name ?"))

[Hello, what, 's, my, name, ?]


In [ ]:
df['tokens'] = df['review'].apply(tokenize)


In [ ]:
df.sample()
type(df['tokens'][0])

list

## Stemming 


In [ ]:
from nltk.stem.porter import PorterStemmer # SnowballStemmer for the Other languages this works well for the english
ps = PorterStemmer()


In [ ]:

def stem_words(tokens):
    # 1. Guard against rows that aren't lists (like NaN or float)
    if not isinstance(tokens, (list, tuple)):
        return []

    # 2. Stringify tokens and skip empty/null items to prevent the .lower() bug
    return [
        ps.stem(str(token))
        for token in tokens
        if token is not None and str(token).strip() != ""
    ]


# Apply the safe function
df["tokens"] = df["tokens"].apply(stem_words)

    # return " ".join([ps.stem(word) for word in text.split()])
test = [ "do ", "does ", "did", "doing" ]
stem_words(test)

['do ', 'does ', 'did', 'do']

In [ ]:
df['tokens'] = df['tokens'].apply(stem_words)


In [ ]:
df.sample()

,id,title,overview,genre,tokens
3580,412202,Handsome Devil,musicmad 16yearold outcast rugbymad boardin...,"['Drama', 'Comedy']","[musicmad, 16yearold, outcast, rugbymad, board..."


## Lemmitizations 

Using WordNet to find the lemma (a word that always exit in language)

Using WordNet from nltk

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()
def lemmatize(tokens):
    
    return [wordnet_lemmatizer.lemmatize(word,pos='v') for word in tokens]

test = ['the', 'mice', 'is', 'running', 'quickly', 'and', 'sees', 'two', 'building', '.']
lemmatize(test)

['the', 'mice', 'be', 'run', 'quickly', 'and', 'see', 'two', 'build', '.']

In [ ]:
df['tokens'] = df['tokens'].apply(lemmatize)


In [ ]:
df.sample()

,id,title,overview,genre,tokens
4033,2140,Kiss of the Dragon,liu jian elite chinese police officer comes ...,"['Action', 'Crime', 'Thriller']","[liu, jian, elit, chine, polic, offic, come, p..."
